# Day 30 — Data Cleaning & Sanitization (Nigerian Retail Sales Analyzer)

This notebook covers ext Normalization, Sentinel Error Correction, Group-Wise Imputation, and Deduplication Pipeline

## Validation goals
- Encapsulate data cleaning steps into a modular, reusable function (`clean_retail_data`)
- Standardize textual columns (`product`, `location`, `supplier`, `category`) using vectorized string methods (`.str.strip().str.title()`)
- Isolate and correct sentinel errors (e.g., negative or zero quantities) by coercing them to `NaN`
- Apply strategic imputation: global median for quantities, category-specific group medians for prices using `.transform()`, and placeholder strings for missing locations
- Export the pristine, analysis-ready dataset to `cleaned_retail_sales.csv` for downstream exploratory data analysis

With our raw multi-region dataset ingested from `raw_retail_sales.csv`, Day 30 focuses on transforming messy, inconsistent inputs into a pristine, analysis-ready table.

Real-world datasets typically feature structural noise such as trailing whitespace, mixed casing, sentinel corruption markers (`e.g., -99 values`), and missing coordinates. We will address these issues using vectorized Pandas operations encapsulated in a modular cleaning function.

### Implementation Script:

In [1]:
import numpy as np
import pandas as pd

# 1. Load Raw Dataset Ingested in Day 29
df_raw = pd.read_csv("raw_retail_sales.csv", parse_dates=["date"])

print("--- Initial Missing Value Audit ---")
print(df_raw.isna().sum())


# ==========================================
# 2. Modular Data Cleaning Pipeline
# ==========================================
def clean_retail_data(df: pd.DataFrame) -> pd.DataFrame:
    df_clean = df.copy()

    # A. Text Normalization: Strip leading/trailing whitespaces and standardize casing
    string_columns = ["product", "location", "supplier", "category"]
    for col in string_columns:
        if col in df_clean.columns:
            df_clean[col] = df_clean[col].str.strip().str.title()

    # B. Sentinel Outlier Correction: Replace negative corruption markers in quantity with NaN
    df_clean.loc[df_clean["quantity"] <= 0, "quantity"] = np.nan

    # C. Missing Value Imputation:
    # - Impute quantity using the column median
    median_qty = df_clean["quantity"].median()
    df_clean["quantity"] = df_clean["quantity"].fillna(median_qty).astype("int32")

    # - Impute price dynamically using group-wise median per product category
    df_clean["price"] = df_clean["price"].fillna(
        df_clean.groupby("category")["price"].transform("median")
    )

    # - Impute location missing values with a placeholder string
    df_clean["location"] = df_clean["location"].fillna("Unknown Location")

    # D. Deduplication: Drop identical row duplicates
    df_clean = df_clean.drop_duplicates()

    return df_clean

# Execute cleaning pipeline
df_cleaned = clean_retail_data(df_raw)

print("\n--- Post-Cleaning Validation ---")
print(df_cleaned.info())
print("\nMissing Values After Cleaning:")
print(df_cleaned.isna().sum())

# Export cleaned artifact for Day 31 Exploratory Data Analysis
df_cleaned.to_csv("cleaned_retail_sales.csv", index=False)
print("\nCleaned dataset successfully saved to 'cleaned_retail_sales.csv'.")

--- Initial Missing Value Audit ---
product      0
price       50
quantity     0
location    30
date         0
supplier     0
category     0
dtype: int64

--- Post-Cleaning Validation ---
<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   product   5000 non-null   str           
 1   price     5000 non-null   float64       
 2   quantity  5000 non-null   int32         
 3   location  5000 non-null   str           
 4   date      5000 non-null   datetime64[us]
 5   supplier  5000 non-null   str           
 6   category  5000 non-null   str           
dtypes: datetime64[us](1), float64(1), int32(1), str(4)
memory usage: 254.0 KB
None

Missing Values After Cleaning:
product     0
price       0
quantity    0
location    0
date        0
supplier    0
category    0
dtype: int64

Cleaned dataset successfully saved to 'cleaned_retail_sales.csv'.
